## Step 8: Detection Methods — CRTAT, Krum, Trimmed Mean, FoolsGold

In [ ]:
# ── 8A. Blockchain Trust (cosine similarity to coordinate-wise median) ──────────
def blockchain_trust(updates):
    """trust_k = cos(u_k, median_j u_j).  The median is taken over ALL participating
    updates of the round (attackers included). `blockchain_trust_loo` below removes
    client k from its own reference; it is used ONLY in the diagnostics cell."""
    stacked = np.stack(updates)
    median  = np.median(stacked, axis=0)
    scores  = []
    for u in updates:
        nu = np.linalg.norm(u); nm = np.linalg.norm(median)
        if nu < 1e-10 or nm < 1e-10: scores.append(0.0)
        else: scores.append(float(np.dot(u, median) / (nu * nm)))
    return np.array(scores)

def blockchain_trust_loo(updates):
    """DIAGNOSTIC ONLY — leave-one-out reference (client k excluded from its own median)."""
    stacked = np.stack(updates); out = []
    for i, u in enumerate(updates):
        ref = np.median(np.delete(stacked, i, axis=0), axis=0)
        nu, nr = np.linalg.norm(u), np.linalg.norm(ref)
        out.append(0.0 if nu < 1e-10 or nr < 1e-10 else float(u @ ref / (nu * nr)))
    return np.array(out)

# ── 8A.2 Inversion-aware signal ────────────────────────────────────────────────
def inversion_signal(updates):
    """cos(u_k, mean_j u_j) over the round's participating updates (attackers included).
    Values near -1 indicate an update pointing against the consensus direction."""
    stacked = np.stack(updates)
    mean    = stacked.mean(axis=0)
    nm      = np.linalg.norm(mean)
    if nm < 1e-10:
        return np.zeros(len(updates))
    out = np.zeros(len(updates))
    for i, u in enumerate(updates):
        nu = np.linalg.norm(u)
        out[i] = 0.0 if nu < 1e-10 else float(np.dot(u, mean) / (nu * nm))
    return out

def inversion_signal_loo(updates):
    """DIAGNOSTIC ONLY — leave-one-out mean reference."""
    stacked = np.stack(updates); out = []
    for i, u in enumerate(updates):
        ref = np.delete(stacked, i, axis=0).mean(axis=0)
        nu, nr = np.linalg.norm(u), np.linalg.norm(ref)
        out.append(0.0 if nu < 1e-10 or nr < 1e-10 else float(u @ ref / (nu * nr)))
    return np.array(out)

# ── 8B. CRTAT — Cross-Round Temporal Anomaly Tracking ──────────────────────────
class CRTAT:
    """
    Cross-Round Temporal Anomaly Tracking — explicit definitions (reviewer response).

    Per client k the tracker keeps a history s_1..s_n of instantaneous trust values
    (trust_k = cosine to the round's coordinate-wise median) and a separate history of
    inversion cosines (observed rounds only). Over the last W = CRTAT_WINDOW history
    entries (n <= W):

      mu_k   = sum_i lam^(n-i) s_i / sum_i lam^(n-i),   lam = CRTAT_DECAY
               ("trust recovery": exponentially weighted mean — older rounds count less;
                lam=1 disables it. NOTE: with lam=0.90 and W=5 the weights span only
                0.66..1.0, so the recovery effect is mild by construction.)
      lam1_k = OLS slope of s against the entries' index 0..n-1        ("trend")
      chg_k  = slope(second half of window) - slope(first half)        ("trend change")
               This is a difference of two slopes in per-index units. It is NOT a
               time-normalised second derivative, so it is not called "acceleration"
               in prose; the constructor argument keeps its historical name `delta`.
      sig_k  = std(s)                                                  ("volatility")
      iota_k = min(0.5, -0.5 * mean(last W inversion cosines)) if that mean < 0 else 0

      A_k = (1 - mu_k) - alpha*lam1_k - delta*chg_k + beta*sig_k + iota_k
      A^_k = min-max normalisation of A over all clients                (in [0, 1])
      D_k = gamma * A^_k + (1 - gamma) * (1 - trust_k / 2)              (final CRTAT score)

    Initialisation: a client with no history gets A = 0.5; with one entry A = 1 - s_1.

    ABSENT CLIENTS (reviewer comment on neutral observations). Only ~40% of clients take
    part in a round. `missing` selects how a non-participating round enters the history:
      'neutral' (original behaviour): the caller inserts NEUTRAL_TRUST (=1.0, "perfectly
                aligned") for that round, so the history is a fixed-length time series
                in which absent rounds look like very trustworthy observations.
      'skip'    (participation-aware): the history holds ONLY rounds in which the client
                actually participated, so no synthetic values enter the slope / volatility.
    """
    def __init__(self, n_clients=NUM_CLIENTS, window=CRTAT_WINDOW,
                 alpha=CRTAT_ALPHA, beta=CRTAT_BETA, gamma=CRTAT_GAMMA,
                 delta=CRTAT_DELTA, decay=CRTAT_DECAY, missing='neutral'):
        assert missing in ('neutral', 'skip')
        self.n, self.window = n_clients, window
        self.alpha, self.beta, self.gamma = alpha, beta, gamma
        self.delta, self.decay, self.missing = delta, decay, missing
        self.history       = defaultdict(list)
        self.participation = defaultdict(int)
        self.inv_history   = defaultdict(list)

    def describe(self):
        return dict(window=self.window, alpha=self.alpha, beta=self.beta, gamma=self.gamma,
                    delta=self.delta, decay=self.decay, missing=self.missing,
                    neutral_value=NEUTRAL_TRUST if self.missing == 'neutral' else None)

    def update(self, trust_scores, participated_mask=None, inv_scores=None):
        """trust_scores: length-n vector (NEUTRAL_TRUST for absent clients when
        missing='neutral'). participated_mask: bool[n]; None = everyone observed."""
        for cid in range(self.n):
            observed = participated_mask is None or bool(participated_mask[cid])
            if observed or self.missing == 'neutral':
                self.history[cid].append(float(trust_scores[cid]))
            if observed:
                self.participation[cid] += 1
                if inv_scores is not None:
                    self.inv_history[cid].append(float(inv_scores[cid]))

    def _trend_change(self, win):
        if len(win) < 4:
            return 0.0
        mid = len(win) // 2
        old, new = win[:mid], win[mid:]
        old_slope = np.polyfit(np.arange(len(old)), old, 1)[0] if len(old) >= 2 else 0.0
        new_slope = np.polyfit(np.arange(len(new)), new, 1)[0] if len(new) >= 2 else 0.0
        return float(new_slope - old_slope)
    _acceleration = _trend_change          # legacy alias

    def _decay_weighted_mean(self, win):
        n = len(win)
        weights = np.array([self.decay**(n-1-i) for i in range(n)])
        return float(np.sum(win * weights) / weights.sum())

    def anomaly_scores(self):
        scores = np.zeros(self.n)
        for cid in range(self.n):
            h = np.array(self.history[cid])
            if len(h) == 0: scores[cid] = 0.5; continue
            if len(h) == 1: scores[cid] = 1 - h[-1]; continue
            win        = h[-self.window:]
            mu         = self._decay_weighted_mean(win)
            trend      = np.polyfit(np.arange(len(win)), win, 1)[0]
            chg        = self._trend_change(win)
            volatility = win.std()
            inv_pen = 0.0
            if self.inv_history[cid]:
                avg_inv = np.array(self.inv_history[cid][-self.window:]).mean()
                if avg_inv < 0:
                    inv_pen = min(0.5, -avg_inv * 0.5)
            scores[cid] = ((1-mu) + self.alpha*(-trend) + self.delta*(-chg)
                           + self.beta*volatility + inv_pen)
        mn, mx = scores.min(), scores.max()
        if mx > mn: scores = (scores-mn)/(mx-mn)
        return scores

    def combined_score(self, trust_scores):
        A_hat   = self.anomaly_scores()
        instant = 1 - np.clip(trust_scores, -1, 1) / 2
        return self.gamma * A_hat + (1-self.gamma) * instant

def crtat_weights(scores, penalty_cap=CRTAT_PENALTY_CAP, temperature=3.0):
    """Aggregation weights actually used by CRTAT+Blockchain: w_k ∝ exp(-3 · cap · D_k).
    Soft down-weighting only — it does NOT bound an update's influence (see the
    influence-bound experiment that adds update scaling and optional norm clipping)."""
    w = np.exp(-temperature * np.asarray(scores, dtype=float) * penalty_cap)
    return w / (w.sum() + 1e-10)

def clip_to_median_norm(updates, ratio=1.0):
    """Explicit influence bound: rescale every update to at most ratio × (median L2 norm
    of the round's updates)."""
    norms = np.array([np.linalg.norm(u) for u in updates])
    tau = ratio * np.median(norms)
    return [u * min(1.0, tau / (n + 1e-12)) for u, n in zip(updates, norms)]

# ── Simplified temporal-consistency baseline (FLDetector-INSPIRED, NOT FLDetector) ──
class TemporalConsistencyBaseline:
    """Cross-round baseline added because the reviewer asks for temporal detectors.
    For every client observed in consecutive participations it measures
        d = ||u_t - u_prev|| / ||u_prev||        (zero-order predictor: 'last update')
    min-max normalised within the round, and scores the client by the mean of its last
    W such values. FLDetector (Zhang et al., KDD 2022) instead predicts the update with an
    L-BFGS Hessian estimate built from global-model history; that is NOT implemented
    here, so this row must not be reported as 'FLDetector' — a faithful implementation
    (or the authors' code) is still needed for the final comparison."""
    def __init__(self, n_clients=NUM_CLIENTS, window=CRTAT_WINDOW):
        self.n, self.window = n_clients, window
        self.prev = {}
        self.obs  = defaultdict(list)
    def update(self, cids, updates):
        d = {}
        for cid, u in zip(cids, updates):
            if cid in self.prev:
                p = self.prev[cid]
                d[cid] = float(np.linalg.norm(u - p) / (np.linalg.norm(p) + 1e-10))
        if d:
            vals = np.array(list(d.values())); lo, hi = vals.min(), vals.max()
            for cid, v in d.items():
                self.obs[cid].append((v - lo) / (hi - lo + 1e-10))
        for cid, u in zip(cids, updates):
            self.prev[cid] = u.copy()
        s = np.full(self.n, np.nan)
        for cid, o in self.obs.items():
            s[cid] = float(np.mean(o[-self.window:]))
        return s

# ── 8C. Multi-Krum ─────────────────────────────────────────────────────────────
def krum_effective_f(n, f):
    """Largest Byzantine bound that satisfies Krum's condition n > 2f + 2.
    With n=12 participants and f=5 the condition fails (12 > 12 is false); we then use
    f_eff = floor((n-3)/2) = 4 and record the clamp (see the protocol-check cell)."""
    return int(max(0, min(f, (n - 3) // 2)))

def krum_select(updates, f=NUM_MALICIOUS):
    n       = len(updates)
    f_used  = krum_effective_f(n, f)
    k       = max(1, n - f_used - 2)
    stacked = np.stack(updates)
    dists   = np.sum((stacked[:,None] - stacked[None,:])**2, axis=-1)
    krum_sc = [sum(sorted(dists[i])[1:k+1]) for i in range(n)]
    n_sel   = max(1, n - f_used - 2)
    selected= np.argsort(krum_sc)[:n_sel]
    w = np.zeros(n); w[selected] = 1.0/n_sel
    return w, np.array(krum_sc)

# ── 8D. Coordinate-wise Trimmed Mean ───────────────────────────────────────────
def trimmed_mean_agg(updates, trim_ratio=0.15):
    stacked = np.stack(updates)
    n, k    = stacked.shape[0], max(1, int(stacked.shape[0]*trim_ratio))
    return np.sort(stacked, axis=0)[k:n-k].mean(0)

# ── 8E. FoolsGold ──────────────────────────────────────────────────────────────
def foolsgold_weights(updates, eps=1e-6):
    S  = np.stack(updates).astype(np.float64)
    norms = np.linalg.norm(S, axis=1, keepdims=True) + eps
    N = (S / norms) @ (S / norms).T
    credit = np.array([max(N[i,j] for j in range(len(S)) if j!=i)
                       for i in range(len(S))])
    w = np.clip(1-credit, 0, 1)
    return w / (w.sum() + eps)

# ── 8F. Median (coordinate-wise) / FedAvg ──────────────────────────────────────
def median_agg(updates):
    return np.median(np.stack(updates), axis=0)

def fedavg_agg(updates, weights=None):
    n = len(updates)
    if weights is None: weights = np.ones(n)/n
    w = np.clip(weights, 0, None); w /= w.sum()+1e-10
    return sum(wi*u for wi,u in zip(w, updates))

# ── Per-client anomaly score used for AUC for EVERY method (reviewer: Table IV) ─
BASELINE_SCORE_DEFINITIONS = {
 'CRTAT+Blockchain' : 'D_k = gamma*A^_k + (1-gamma)*(1 - trust_k/2); trust_k = cos(u_k, coordinate-wise median of the round).',
 'Krum'             : 'Krum score = sum of squared L2 distances to the n-f_eff-2 nearest updates, min-max normalised within the round (f_eff = min(f, floor((n-3)/2)) so n > 2f+2 holds).',
 'TrimmedMean'      : 'L2 distance of u_k to the coordinate-wise trimmed-mean aggregate (15% trimmed per tail), min-max normalised within the round.',
 'FoolsGold'        : '1 - w_k / max_j w_j with w_k = clip(1 - max_{j!=k} cos(u_k,u_j), 0, 1) (normalised).',
 'CoordMedian'      : 'L2 distance of u_k to the coordinate-wise median aggregate, min-max normalised within the round.',
 'FedAvg_NoDefense' : 'FedAvg is NOT a detector. Proxy: 1 - (cos(u_k, median)+1)/2 on FedAvg\'s own updates (the same cosine-to-median trust as CRTAT\'s instantaneous signal, computed on FedAvg\'s own global model).',
 'TemporalConsistency': 'Simplified FLDetector-INSPIRED baseline (zero-order predictor, no L-BFGS): windowed mean of normalised ||u_t - u_prev|| / ||u_prev|| on CRTAT-branch updates. Not FLDetector.',
}
def print_baseline_definitions():
    print('Per-client anomaly score derived for each method (higher = more suspicious):')
    for m, d in BASELINE_SCORE_DEFINITIONS.items():
        print(f'  • {m:<18}: {d}')

print('✅ Detection & aggregation methods defined (v9).')
print('   CRTAT+Blockchain   [proposed — explicit definitions, participation-aware option]')
print('   Krum               [Blanchard et al., 2017 — f clamped to satisfy n > 2f+2]')
print('   Trimmed Mean / Coordinate Median [Yin et al., 2018]')
print('   FoolsGold          [Fung et al., 2018]')
print('   FedAvg NoDefense   [McMahan et al., 2017]')
print('   TemporalConsistency [simplified FLDetector-inspired baseline — NOT FLDetector]')
print()
print_baseline_definitions()


✅ Detection & aggregation methods defined (v9).
   CRTAT+Blockchain   [proposed — explicit definitions, participation-aware option]
   Krum               [Blanchard et al., 2017 — f clamped to satisfy n > 2f+2]
   Trimmed Mean / Coordinate Median [Yin et al., 2018]
   FoolsGold          [Fung et al., 2018]
   FedAvg NoDefense   [McMahan et al., 2017]
   TemporalConsistency [simplified FLDetector-inspired baseline — NOT FLDetector]

Per-client anomaly score derived for each method (higher = more suspicious):
  • CRTAT+Blockchain  : D_k = gamma*A^_k + (1-gamma)*(1 - trust_k/2); trust_k = cos(u_k, coordinate-wise median of the round).
  • Krum              : Krum score = sum of squared L2 distances to the n-f_eff-2 nearest updates, min-max normalised within the round (f_eff = min(f, floor((n-3)/2)) so n > 2f+2 holds).
  • TrimmedMean       : L2 distance of u_k to the coordinate-wise trimmed-mean aggregate (15% trimmed per tail), min-max normalised within the round.
  • FoolsGold         :